# Sensitivity of filter feeders to macroaggregate sinking (`p_fMAsink`)

Runs GOTM-BFM for the OysterGrounds case once per value of `p_fMAsink`
(scaling factor for the macroaggregate sinking rate; 1 = unchanged model),
then plots the mean suspension-feeder biomass (`Y3c`) over one year:

* **(a)** annual mean `Y3c` against `p_fMAsink`, labelled with the % change from `p_fMAsink = 1`
* **(b)** daily `Y3c` through the year, one line per `p_fMAsink`

One year takes about 6 minutes per run; the runs are started in parallel.

In [ ]:
# Settings
P_FMASINK_VALUES = [1, 2, 3, 4, 5, 6]
START = "1972-01-01 00:00:00"
STOP = "1973-01-01 00:00:00"
YEAR = 1972            # year to average over
VAR = "Y3c"            # BFM variable in OG.nc, e.g. "jnetY3c" for net production
RESULTS_ROOT = "results/fMAsink"
MAX_PARALLEL = 6       # runs at the same time

In [ ]:
# GOTM runner
import os
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path


def run_gotm(p_fMAsink, start=START, stop=STOP, root=RESULTS_ROOT):
    """One GOTM-BFM run with the given p_fMAsink; returns (value, returncode, run dir)."""
    results_dir = Path(root) / "fMAsink_{:g}".format(p_fMAsink)
    if results_dir.exists():          # gotm-run needs a new or empty directory
        shutil.rmtree(str(results_dir))
    results_dir.parent.mkdir(parents=True, exist_ok=True)

    command = [
        "gotm-run",
        "--out", str(results_dir),
        "--start", start,
        "--stop", stop,
        "--set", "PelGlobal.nml:p_fMAsink={}".format(p_fMAsink),
    ]
    proc = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return p_fMAsink, proc.returncode, str(results_dir)


print("Starting {} runs from {} to {} ...".format(len(P_FMASINK_VALUES), START, STOP))
with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as pool:
    runs = list(pool.map(run_gotm, P_FMASINK_VALUES))

for value, returncode, run_dir in runs:
    status = "finished" if returncode == 0 else "FAILED (see {}/log.out)".format(run_dir)
    print("p_fMAsink = {:<4g} {}".format(value, status))

In [ ]:
# Read the results: annual mean of VAR per run
import datetime
import netCDF4
import numpy as np


def read_year(path, var, year):
    """Dates and values of `var` within `year`, plus its units and long name."""
    with netCDF4.Dataset(path) as nc:
        t = nc.variables["time"]
        # num2date may return cftime dates, which matplotlib cannot plot:
        # convert them to standard Python datetimes
        dates = [datetime.datetime(d.year, d.month, d.day, d.hour, d.minute, d.second)
                 for d in netCDF4.num2date(t[:], t.units)]
        values = np.ma.filled(nc.variables[var][:].reshape(len(dates), -1)[:, 0], np.nan)
        units = getattr(nc.variables[var], "units", "")
        long_name = getattr(nc.variables[var], "long_name", var)
    keep = np.array([d.year == year for d in dates])
    return [d for d, k in zip(dates, keep) if k], values[keep], units, long_name


results = []    # (p_fMAsink, dates, values)
for value, returncode, run_dir in runs:
    if returncode != 0:
        continue
    dates, values, units, long_name = read_year(os.path.join(run_dir, "OG.nc"), VAR, YEAR)
    results.append((value, dates, values))
results.sort(key=lambda r: r[0])

fs = np.array([r[0] for r in results], dtype=float)
means = np.array([np.nanmean(r[2]) for r in results])
ref = means[np.argmin(abs(fs - 1.0))]          # p_fMAsink = 1 (or the closest value)
change = 100.0 * (means - ref) / ref

print("{:>9}  {:>12}  {:>8}".format("p_fMAsink", "mean " + VAR, "change"))
for f, m, c in zip(fs, means, change):
    print("{:>9g}  {:>12.2f}  {:>+7.2f}%".format(f, m, c))

In [ ]:
# Plot
%matplotlib inline
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(results)))

ax1.plot(fs, means, "-", color="0.6", zorder=1)
ax1.scatter(fs, means, c=colors, s=50, zorder=2)
for f, m, c in zip(fs, means, change):
    ax1.annotate("{:+.1f}%".format(c), (f, m), textcoords="offset points",
                 xytext=(0, 8), ha="center", fontsize=9)
ax1.set_xlabel("p_fMAsink (macroaggregate sinking scaling factor)")
ax1.set_ylabel("annual mean {} [{}]".format(VAR, units))
ax1.set_title("(a) {}, mean over {}".format(long_name, YEAR), fontsize=10)
ax1.set_xticks(fs)
ax1.margins(y=0.12)

for (f, dates, values), col in zip(results, colors):
    ax2.plot(dates, values, color=col, label="{:g}".format(f))
ax2.set_ylabel("{} [{}]".format(VAR, units))
ax2.set_title("(b) daily values in {}".format(YEAR), fontsize=10)
ax2.legend(title="p_fMAsink", fontsize=8, frameon=False)
fig.autofmt_xdate()
fig.tight_layout()

fig.savefig(os.path.join(RESULTS_ROOT, "fMAsink_{}_{}.png".format(VAR, YEAR)), dpi=150)
plt.show()